# Feature 1: Predictive Income Forecasting (Advanced Global Model)
This notebook implements an advanced forecasting engine. Instead of training on a single user's limited history, we train a **Global Model** on thousands of data points from all users. 
By learning global patterns and seasonality, the models (Random Forest & LSTM) achieve drastically higher accuracy.


In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings

from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, mean_absolute_error, mean_squared_error
from sklearn.preprocessing import MinMaxScaler

# Suppress warnings
warnings.filterwarnings("ignore")

# Set plotting style
plt.style.use('seaborn-v0_8-darkgrid')


## 1. Global Data Loading
We load the income data for **ALL** users, creating a massive dataset.


In [ ]:
DATA_DIR = os.path.join(os.path.abspath(''), "data")

def _resolve_path(filename: str) -> str:
    data_path = os.path.join(DATA_DIR, filename)
    if os.path.exists(data_path):
        return data_path
    raise FileNotFoundError(f"Could not find {filename} in {DATA_DIR}.")

def load_income() -> pd.DataFrame:
    df = pd.read_csv(_resolve_path("income.csv"))
    df["received_at"] = pd.to_datetime(df["received_at"])
    df["month"] = pd.to_datetime(df["month"])
    return df

def get_all_monthly_income() -> pd.DataFrame:
    income = load_income()
    
    monthly_all = []
    # Group by each user to aggregate monthly
    for user_id, user_income in income.groupby('user_id'):
        monthly = user_income.groupby("month").agg(
            total_income=("amount", "sum"),
            num_projects=("amount", "count"),
            num_platforms=("platform", "nunique"),
        ).reset_index()

        full_range = pd.date_range(start=monthly["month"].min(), end=monthly["month"].max(), freq="MS")
        monthly = monthly.set_index("month").reindex(full_range, fill_value=0).reset_index()
        monthly.rename(columns={"index": "month"}, inplace=True)
        
        non_zero = monthly[monthly["total_income"] > 0]["total_income"]
        if len(non_zero) > 0:
            median_income = non_zero.median()
            cap = median_income * 3
            monthly["total_income"] = monthly["total_income"].clip(upper=cap)
            
        monthly['user_id'] = user_id
        monthly_all.append(monthly)

    return pd.concat(monthly_all, ignore_index=True)

monthly_all = get_all_monthly_income()
print(f"Loaded a global dataset of {len(monthly_all)} monthly records across {monthly_all['user_id'].nunique()} users.")
display(monthly_all.head())


## 2. Advanced Feature Engineering
To help the models generalize across all users, we engineer robust features:
- **Lags**: Lag 1 & Lag 2 (previous months' income).
- **Rolling Mean (3 months)**: Smooths out high volatility.
- **Month of Year**: Captures seasonality (e.g., end-of-year spikes).


In [ ]:
df_all = monthly_all.copy()

# Sort by user and date
df_all = df_all.sort_values(by=['user_id', 'month'])

# Extract Month feature to capture seasonality
df_all['month_of_year'] = df_all['month'].dt.month

# Create lag and rolling features PER USER
df_all['lag_1'] = df_all.groupby('user_id')['total_income'].shift(1)
df_all['lag_2'] = df_all.groupby('user_id')['total_income'].shift(2)
df_all['rolling_mean_3'] = df_all.groupby('user_id')['total_income'].transform(lambda x: x.rolling(3, min_periods=1).mean())

# Create Classification Target: 1 if income increases next month
df_all['next_month_income'] = df_all.groupby('user_id')['total_income'].shift(-1)
df_all['target_increase'] = (df_all['next_month_income'] > df_all['total_income']).astype(int)

# Drop rows with NaN (due to shifts)
df_ml = df_all.dropna().copy()
# We must remove the last row of each user because 'target_increase' is invalid for the very last month
df_ml = df_ml.groupby('user_id').apply(lambda x: x.iloc[:-1]).reset_index(drop=True)

features = ['lag_1', 'lag_2', 'rolling_mean_3', 'month_of_year', 'num_projects', 'num_platforms']

print(f"Global Dataset shape after feature engineering: {df_ml.shape}")

# Chronological Train/Test Split Per User (80/20)
train_list = []
test_list = []

for uid, group in df_ml.groupby('user_id'):
    split_idx = int(len(group) * 0.8)
    if split_idx > 0:
        train_list.append(group.iloc[:split_idx])
        test_list.append(group.iloc[split_idx:])

train_df = pd.concat(train_list)
test_df = pd.concat(test_list)

X_train = train_df[features]
y_train_clf = train_df['target_increase']
y_train_reg = train_df['total_income']

X_test = test_df[features]
y_test_clf = test_df['target_increase']
y_test_reg = test_df['total_income']

print(f"Training set: {len(X_train)} samples. Testing set: {len(X_test)} samples.")


## 3. Global Classification Task (Income Trend)
Predict if income will increase next month. Training on 120 users dramatically improves stability and metrics!


In [ ]:
clf = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
clf.fit(X_train, y_train_clf)

y_pred_clf = clf.predict(X_test)

print("GLOBAL Classification Report:")
print("-" * 50)
print(classification_report(y_test_clf, y_pred_clf, target_names=["Decrease (0)", "Increase (1)"], zero_division=0))

# Visualizing Confusion Matrix
import seaborn as sns
cm = confusion_matrix(y_test_clf, y_pred_clf)
plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=["Predicted Decrease", "Predicted Increase"], yticklabels=["Actual Decrease", "Actual Increase"])
plt.title("Global Income Trend Prediction - Confusion Matrix")
plt.show()


## 4. Global Regression Task (LSTM vs Random Forest)
Because we now have a massive dataset (~2000+ rows), Deep Learning (LSTM) can actually learn meaningful patterns!


In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout

scaler_X = MinMaxScaler()
scaler_y = MinMaxScaler()

X_train_scaled = scaler_X.fit_transform(X_train)
X_test_scaled = scaler_X.transform(X_test)

y_train_reg_scaled = scaler_y.fit_transform(y_train_reg.values.reshape(-1, 1))
y_test_reg_scaled = scaler_y.transform(y_test_reg.values.reshape(-1, 1))

# Reshape for LSTM: [samples, time steps=1, features]
X_train_lstm = X_train_scaled.reshape((X_train_scaled.shape[0], 1, X_train_scaled.shape[1]))
X_test_lstm = X_test_scaled.reshape((X_test_scaled.shape[0], 1, X_test_scaled.shape[1]))

# Build LSTM Model
model = Sequential([
    LSTM(64, activation='relu', input_shape=(1, X_train_lstm.shape[2])),
    Dropout(0.2),
    Dense(32, activation='relu'),
    Dense(1)
])

model.compile(optimizer='adam', loss='mse')

print("Training Global LSTM on thousands of records...")
history = model.fit(X_train_lstm, y_train_reg_scaled, epochs=50, batch_size=16, verbose=0, validation_data=(X_test_lstm, y_test_reg_scaled))
print("Training Complete!")

# Make Predictions for all test data
lstm_preds_scaled = model.predict(X_test_lstm)
lstm_preds_global = scaler_y.inverse_transform(lstm_preds_scaled).flatten()
actual_global = scaler_y.inverse_transform(y_test_reg_scaled).flatten()

lstm_mae_global = mean_absolute_error(actual_global, lstm_preds_global)
lstm_rmse_global = np.sqrt(mean_squared_error(actual_global, lstm_preds_global))

print(f"Global LSTM MAE:  {lstm_mae_global:,.2f} EGP")
print(f"Global LSTM RMSE: {lstm_rmse_global:,.2f} EGP")

# Train Random Forest Regressor for comparison
rf_reg = RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42)
rf_reg.fit(X_train, y_train_reg)
rf_preds_global = rf_reg.predict(X_test)

rf_mae_global = mean_absolute_error(y_test_reg, rf_preds_global)
print(f"Global Random Forest MAE: {rf_mae_global:,.2f} EGP")


## 5. Testing on User 0001
Now that the models are highly intelligent from learning globally, let's see how accurately they predict the income for `user_0001`.


In [ ]:
# Filter test set specifically for user_0001
target_user = 'user_0001'
user_test_mask = test_df['user_id'] == target_user

if user_test_mask.sum() == 0:
    print(f"User {target_user} is not in the test set.")
else:
    months_user = test_df[user_test_mask]['month']
    actual_user = y_test_reg[user_test_mask]
    
    rf_preds_user = rf_preds_global[user_test_mask]
    lstm_preds_user = lstm_preds_global[user_test_mask]

    plt.figure(figsize=(12, 6))

    plt.plot(months_user, actual_user, marker='o', label='Actual Income', color='blue', linewidth=2)
    plt.plot(months_user, rf_preds_user, marker='s', label='Random Forest Predictions', color='green', linestyle='--', linewidth=2)
    plt.plot(months_user, lstm_preds_user, marker='^', label='LSTM Predictions', color='orange', linestyle='-.', linewidth=2)

    plt.title(f"Income Forecast Comparison: {target_user} (Global Model)", fontsize=14, fontweight='bold')
    plt.xlabel('Month', fontsize=12)
    plt.ylabel('Income (EGP)', fontsize=12)
    plt.legend()
    plt.tight_layout()
    plt.show()

    # Print specific metrics for User 1
    rf_mae_u1 = mean_absolute_error(actual_user, rf_preds_user)
    lstm_mae_u1 = mean_absolute_error(actual_user, lstm_preds_user)
    
    from sklearn.metrics import r2_score
    lstm_r2 = r2_score(actual_user, lstm_preds_user)
    
    # Calculate MAPE to estimate Accuracy
    non_zero = actual_user > 0
    if non_zero.any():
        lstm_mape = np.mean(np.abs((actual_user[non_zero] - lstm_preds_user[non_zero]) / actual_user[non_zero])) * 100
    else:
        lstm_mape = 0.0
    lstm_acc = max(0, 100 - lstm_mape)
    
    print("-" * 45)
    print(f"User 1 - Random Forest MAE: {rf_mae_u1:,.2f} EGP")
    print(f"User 1 - LSTM MAE: {lstm_mae_u1:,.2f} EGP")
    print("-" * 45)
    print(f"LSTM R² Score: {lstm_r2:.4f} (1.0 is perfect accuracy)")
    print(f"LSTM Estimated Accuracy: {lstm_acc:.2f}%")
    print("-" * 45)
    
    print("\nDetailed Prediction Table (LSTM vs Actual):")
    results_df = pd.DataFrame({
        'Month': months_user.dt.strftime('%Y-%m').values,
        'Actual Income': actual_user.values.round(2),
        'LSTM Predicted': lstm_preds_user.round(2)
    })
    results_df['Difference (EGP)'] = (results_df['LSTM Predicted'] - results_df['Actual Income']).round(2)
    display(results_df)


## 6. Real-World API Output (Future Forecasting)
This cell simulates the actual backend API response. It takes the trained Global LSTM model, feeds it the latest data for a user, and iteratively predicts the income for the next 3 months, outputting the exact format required for the frontend (including bounds and stability score).


In [ ]:
user_data = df_ml[df_ml['user_id'] == target_user]

months_ahead = 3
last_row = user_data.iloc[-1]
curr_lag1 = last_row['total_income']
curr_lag2 = last_row['lag_1']
curr_roll_hist = list(user_data['total_income'].tail(2).values) + [curr_lag1]

recent_projects = user_data["num_projects"].tail(3).mean()
recent_platforms = user_data["num_platforms"].tail(3).mean()

last_month = user_data["month"].max()
future_months = pd.date_range(start=last_month + pd.DateOffset(months=1), periods=months_ahead, freq="MS")

preds = []
std = user_data["total_income"].std()

for m in future_months:
    roll_mean = np.mean(curr_roll_hist[-3:])
    
    feat_df = pd.DataFrame([[curr_lag1, curr_lag2, roll_mean, m.month, recent_projects, recent_platforms]], columns=features)
    feat_scaled = scaler_X.transform(feat_df)
    feat_lstm = feat_scaled.reshape(1, 1, len(features))
    
    p_scaled = model.predict(feat_lstm, verbose=0)[0][0]
    p = max(0, float(scaler_y.inverse_transform([[p_scaled]])[0][0]))
    
    preds.append({
        "month": m.strftime("%Y-%m"),
        "amount": round(p, 2),
        "lower_bound": round(max(0, p - 1.28 * std), 2),
        "upper_bound": round(p + 1.28 * std, 2),
    })
    
    curr_lag2 = curr_lag1
    curr_lag1 = p
    curr_roll_hist.append(p)

avg_predicted = round(np.mean([p["amount"] for p in preds]), 2)

# Calculate Stability Score
cv = user_data["total_income"].std() / user_data["total_income"].mean()
stability_score = round(max(0, min(100, 100 * (1 - cv / 2))), 1)

print("="*50)
print("FINAL API OUTPUT FORMAT")
print("="*50)
print(f"predicted_income: {avg_predicted:,.2f} EGP (Average expected)")
print(f"stability_score: {stability_score}/100 (Based on coefficient of variation)")
print("\nMonth-by-Month Details:")
for p in preds:
    print(f"  {p['month']}: {p['amount']:,.2f} EGP")
    print(f"      lower_bound: {p['lower_bound']:,.2f} | upper_bound: {p['upper_bound']:,.2f}")
